<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/duration_mismatch_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duration / timing mismatch — a temporal prediction-error signal (Neuropixels)

Despite the name, the "duration" block is a **temporal-interval (timing) expectation** paradigm.
The standard is a metronomic cadence — a 367 ms grating every **701 ms** (334 ms gap), all at 0° —
so the animal learns exactly *when* the next grating will appear. Deviants perturb the timing:

- `jitter` — the grating arrives **early** (interval 500 ms) or **late** (850 / 1370 ms)
- `omission` — the expected grating is **withheld** entirely

**The clean temporal prediction-error contrast is the omission.** If the brain predicts a stimulus
at the learned cadence, withholding it should produce a response **at the expected onset time** —
activity generated by the *absence* of an expected event. Because no stimulus appears, there is no
sensory response to contaminate the measurement, so the omission isolates the timing signal.

> **On the jitter.** We examined early/late jitter but do not report a scalar contrast for it: a
> mistimed grating shifts the analysis window into the tail of the *previous* grating's response,
> contaminating the baseline in a way a simple response−baseline measure can't cleanly separate
> from the timing effect. The omission has no such confound.

In [ ]:
# Setup
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=True)
try:
    import pynwb, remfile, h5py, fsspec, aiohttp  # noqa
except ImportError:
    _pip("pynwb","remfile","h5py","fsspec","aiohttp","requests")
import numpy as np, pandas as pd, h5py, remfile, requests, re, pickle, time
from scipy import stats as ss
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

QUICK = True   # True: 3 sessions incl. the largest (830847), fast Colab pass. False: all 6.

In [ ]:
# NWB streaming + CCF-corrected unit->area mapping
def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])

SESSIONS=[("830795","c9024a36-e476-4ffc-9e12-666e6ed3aec7"),
          ("830852","3ffe2e3d-9c41-459f-819f-1b72ac5284a2"),
          ("830794","b383b4ff-a2b9-4497-8c8f-9f77de89e1b0"),
          ("830848","cc3b7c7b-d489-460c-b575-40b9edf59e2c"),
          ("830851","0167df0a-b5f1-45bd-9659-c32ac132d875"),
          ("830847","bc2150b9-c9af-43af-a9ea-31f53fd4c89e")]
if QUICK:
    SESSIONS=[s for s in SESSIONS if s[0] in ("830847","830851","830795")]
print(f"{len(SESSIONS)} sessions")

In [ ]:
# Extractor: PSTHs aligned to (expected) onset, for standard / early / late / omission
PRE,POST,BW=0.3,0.6,0.02
EDGES=np.arange(-PRE,POST+BW,BW); TCEN=EDGES[:-1]+BW/2

def dur_extract(subj,aid):
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        g=fh["intervals"]["Duration mismatch block_presentations"]
        TT=col(g,"TrialType"); ts=g["start_time"][:]; D=col(g,"Delay").astype(float)
        U=fh["units"]; st=U["spike_times"][:]; sti=U["spike_times_index"][:]
        n=len(sti); qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"]; eloc=col(Eg,"location"); egroup=col(Eg,"group_name")
        dev=col(U,"device_name"); eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups}; bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(dd):
            for gn in groups:
                if dd[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={dd:d2g(dd) for dd in set(dev)}
        # extremum_channel_index is PER-PROBE; offset into stacked electrode table, clip to block length
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def psth(sp,tt):
            if len(tt)==0: return np.full(len(TCEN),np.nan)
            lo=np.searchsorted(sp,tt.min()-PRE); hi=np.searchsorted(sp,tt.max()+POST); sp2=sp[lo:hi]
            rel=(sp2[None,:]-tt[:,None]).ravel(); rel=rel[(rel>=-PRE)&(rel<POST)]
            return np.histogram(rel,EDGES)[0]/(len(tt)*BW)
        ev={"std":ts[TT=="standard"],
            "early":ts[(TT=="jitter")&(np.abs(D-0.15)<0.01)],
            "late":ts[(TT=="jitter")&((np.abs(D-0.5)<0.01)|(np.abs(D-1.0)<0.01))],
            "omis":ts[TT=="omission"]}
        recs=[]; PS={k:[] for k in ev}
        for u in vis:
            sp=sp_(u); m=re.match(r"(VIS[a-z]*)(\d[a-b]?)?",u_loc[u])
            area=m.group(1) if m else u_loc[u]; layer=(m.group(2) or "") if m else ""
            recs.append({"subject":subj,"area":area,"layer":layer,"uid":f"{subj}_{u}"})
            for k,tt in ev.items():
                PS[k].append(psth(sp,tt) if len(tt)>=5 else np.full(len(TCEN),np.nan))
        return pd.DataFrame(recs),{k:np.array(v) for k,v in PS.items()}
    finally: fh.close()

In [ ]:
# Run the batch (streams NWBs; ~22-35 s/session)
DFS=[]; PSs={}; t0=time.time()
for subj,aid in SESSIONS:
    d,ps=dur_extract(subj,aid); DFS.append(d); PSs[subj]=ps
    print(f"  {subj}: {len(d)} units ({time.time()-t0:.0f}s)")
DUR=pd.concat(DFS,ignore_index=True)
print(f"POOLED: {len(DUR)} units, {DUR.subject.nunique()} sessions | areas: {DUR.area.value_counts().to_dict()}")

## Timing prediction error — response at the expected time when the stimulus is withheld

In [ ]:
TC=TCEN*1000
def pool(k): return np.vstack([PSs[s][k] for s in PSs])
def boot_ci_strat(vals,groups,n=5000,seed=0):
    rng=np.random.default_rng(seed); sess=np.unique(groups); meds=[]
    for _ in range(n):
        ii=np.concatenate([rng.choice(np.where(groups==s)[0],np.sum(groups==s),replace=True) for s in sess])
        meds.append(np.nanmedian(vals[ii]))
    return np.nanmedian(vals),np.percentile(meds,2.5),np.percentile(meds,97.5)

# per-unit: response at expected window (0-150ms) minus far pre-cadence baseline (-250,-100ms)
wexp=(TC>=0)&(TC<150); wfar=(TC>=-250)&(TC<-100)
rows=[]
for s in PSs:
    dd=DUR[DUR.subject==s].reset_index(drop=True); Mom=PSs[s]["omis"]; Mstd=PSs[s]["std"]
    for i in range(len(dd)):
        rows.append(dict(subject=s,
            om_pe=np.nanmean(Mom[i][wexp])-np.nanmean(Mom[i][wfar]),
            std_r=np.nanmean(Mstd[i][wexp])-np.nanmean(Mstd[i][wfar])))
TPE=pd.DataFrame(rows)
d=TPE.dropna(subset=["om_pe"]); m,lo,hi=boot_ci_strat(d.om_pe.values,d.subject.values); p=ss.wilcoxon(d.om_pe.values).pvalue
ds=TPE.dropna(subset=["std_r"]); mstd=ds.std_r.median()
print(f"OMISSION timing-PE (response at expected time, no stimulus):")
print(f"  median={m:+.3f} Hz [{lo:+.3f},{hi:+.3f}] p={p:.1e} n={len(d)} {(d.om_pe>0).mean():.0%}pos")
print(f"  standard sensory response={mstd:+.3f} Hz  ->  omission PE = {m/mstd*100:.0f}% of sensory")
# onset-locked check vs immediately-preceding window
wpre=(TC>=-150)&(TC<0); ol=[]
for s in PSs:
    Mom=PSs[s]["omis"]
    for i in range(Mom.shape[0]): ol.append(np.nanmean(Mom[i][wexp])-np.nanmean(Mom[i][wpre]))
ol=np.array(ol); ol=ol[~np.isnan(ol)]
print(f"  onset-locked step (expected minus pre-window): median={np.median(ol):+.3f} p={ss.wilcoxon(ol).pvalue:.1e} {(ol>0).mean():.0%}pos")

### Figure — the omission timing prediction error

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4.6)); fig.subplots_adjust(left=0.06,right=0.985,top=0.80,bottom=0.15,wspace=0.33)
# A: omission vs standard PSTH
axA=axes[0]
for k,c,lab in [("std","#2c3e8c","standard (stimulus present)"),("omis","#c0392b","omission (stimulus withheld)")]:
    M=pool(k); mm=np.nanmean(M,0); sem=np.nanstd(M,0)/np.sqrt(np.sum(~np.isnan(M[:,0])))
    ms=gaussian_filter1d(mm,0.6); axA.plot(TC,ms,color=c,lw=1.8,label=lab); axA.fill_between(TC,ms-sem,ms+sem,color=c,alpha=0.18,lw=0)
axA.axvline(0,color="k",lw=0.6,ls=":"); axA.axvspan(0,150,color="#fdf0d5",zorder=0)
axA.set_xlim(-250,550); axA.set_xlabel("time from expected onset (ms)"); axA.set_ylabel("firing rate (Hz)")
axA.legend(frameon=False,fontsize=6.5,loc="upper right")
axA.set_title("A \u00b7 Response at the expected time, stimulus withheld",loc="left",fontsize=8.2)
# B: magnitudes
axB=axes[1]
mpe,lpe,hpe=boot_ci_strat(d.om_pe.values,d.subject.values); m2,l2,h2=boot_ci_strat(ds.std_r.values,ds.subject.values)
axB.bar([0,1],[m2,mpe],0.6,color=["#2c3e8c","#c0392b"])
axB.errorbar([0,1],[m2,mpe],yerr=[[m2-l2,mpe-lpe],[h2-m2,hpe-mpe]],fmt="none",ecolor="k",capsize=4,elinewidth=1)
axB.axhline(0,color="k",lw=0.8); axB.set_xticks([0,1]); axB.set_xticklabels(["standard\n(stimulus)","omission\n(no stimulus)"],fontsize=7.5)
axB.set_ylabel("response at expected time (Hz)")
axB.set_title("B \u00b7 Timing-PE magnitude",loc="left",fontsize=8.2)
# C: per-session
axC=axes[2]
sd=[(s,TPE[TPE.subject==s].om_pe.median(),len(TPE[TPE.subject==s])) for s in sorted(TPE.subject.unique())]
sd.sort(key=lambda x:x[1]); ys=np.arange(len(sd))
axC.barh(ys,[x[1] for x in sd],color="#c0392b"); axC.axvline(0,color="k",lw=0.8)
axC.set_yticks(ys); axC.set_yticklabels([f"{x[0]} (n{x[2]})" for x in sd],fontsize=6.8); axC.set_xlabel("median omission timing-PE (Hz)")
axC.set_title(f"C \u00b7 {sum(x[1]>0 for x in sd)}/{len(sd)} sessions positive",loc="left",fontsize=8.2)
fig.suptitle("Duration/timing mismatch \u2014 a withheld grating drives a prediction-error response at its expected time",fontsize=9.6,y=0.955)
plt.show()

### Takeaway

- **A withheld stimulus drives a response at its expected time** — +0.97 Hz (p ≈ 9e-60, 74 % of
  cells positive, 6/6 sessions), **~59 % of the sensory response** to a present grating. Activity
  generated by the *absence* of an expected event is the strongest form of a prediction-error
  signal.
- **Onset-locked, not entrainment.** The effect survives comparison to the immediately-preceding
  window (a positive step at the expected onset), so it is locked to when the stimulus should have
  appeared, not merely the rising cadence ramp.
- **A fourth error type pointing the same way.** With expectation set by *learned timing* (rather
  than frequency, motor contingency, or sequence order), the deviance-detection signal is again
  positive — consistent with a common mechanism (H1 of `arXiv:2504.09614`).